# Q5 - Real-world Dataset 2 (spotify_dataset.csv)
วิเคราะห์แบบแยกจาก bread_basket โดยใช้ user เป็น transaction และ artist เป็น item

In [1]:
import warnings

import pandas as pd
from mlxtend.frequent_patterns import apriori, association_rules
from scipy.stats import chi2_contingency

warnings.filterwarnings("ignore")
pd.set_option("display.max_rows", 200)
pd.set_option("display.width", 170)

In [2]:
# The full file is very large; this notebook intentionally uses a large sample window.
sample_rows = 200_000
raw = pd.read_csv("dataset/spotify_dataset.csv", on_bad_lines="skip", nrows=sample_rows)
raw.columns = raw.columns.str.strip().str.replace('"', "", regex=False)

for col in raw.columns:
    if raw[col].dtype == "object":
        raw[col] = raw[col].str.replace('"', "", regex=False).str.strip()

print("Loaded rows:", len(raw))
print("Unique users:", raw["user_id"].nunique())
print("Unique artists:", raw["artistname"].nunique())
print("\nTop 15 artists:")
print(raw["artistname"].value_counts().head(15))

Loaded rows: 200000
Unique users: 239
Unique artists: 20202

Top 15 artists:
artistname
Johnny Cash              1026
Red Hot Chili Peppers    1000
Arctic Monkeys            775
Daft Punk                 762
Pearl Jam                 731
JAY Z                     662
Andrés Calamaro           610
Ramones                   573
Queen                     565
Coldplay                  558
Blur                      511
Nirvana                   497
Green Day                 495
Kanye West                493
Bruce Springsteen         467
Name: count, dtype: int64


In [3]:
top_n_artists = 30
top_artists = raw["artistname"].value_counts().head(top_n_artists).index
filtered = raw[raw["artistname"].isin(top_artists)].copy()

basket = (
    filtered.groupby(["user_id", "artistname"])["artistname"]
    .count()
    .unstack(fill_value=0)
    .gt(0)
)

print("Filtered rows (top 30 artists):", len(filtered))
print("Basket shape:", basket.shape)

support_grid = [0.05, 0.10, 0.15, 0.20, 0.25]
support_scan_rows = []
for ms in support_grid:
    fi = apriori(basket, min_support=ms, use_colnames=True, max_len=3)
    support_scan_rows.append([ms, len(fi), int((fi["itemsets"].apply(len) >= 2).sum())])

support_scan = pd.DataFrame(
    support_scan_rows,
    columns=["min_support", "all_frequent_itemsets", "frequent_itemsets_ge2"],
)
print("\nSupport scan:")
print(support_scan.to_string(index=False))

Filtered rows (top 30 artists): 15969
Basket shape: (184, 30)

Support scan:
 min_support  all_frequent_itemsets  frequent_itemsets_ge2
        0.05                    848                    821
        0.10                    140                    115
        0.15                     42                     20
        0.20                     18                      1
        0.25                     13                      0


In [4]:
min_sup = 0.15
min_conf = 0.40

fi = apriori(basket, min_support=min_sup, use_colnames=True, max_len=3)
rules = association_rules(
    fi,
    metric="support",
    min_threshold=min_sup,
    num_itemsets=len(fi),
)

rules_A = rules[(rules["support"] >= min_sup) & (rules["confidence"] >= min_conf)].copy()
rules_B = rules_A[rules_A["lift"] > 1].copy()

def p_value_for_rule(row):
    antecedent = list(row["antecedents"])
    consequent = list(row["consequents"])
    x = basket[antecedent].all(axis=1)
    y = basket[consequent].all(axis=1)
    n11 = int(((x) & (y)).sum())
    n10 = int(((x) & (~y)).sum())
    n01 = int(((~x) & (y)).sum())
    n00 = int(((~x) & (~y)).sum())

    row1 = n11 + n10
    row2 = n01 + n00
    col1 = n11 + n01
    col2 = n10 + n00
    if row1 == 0 or row2 == 0 or col1 == 0 or col2 == 0:
        return 1.0

    _, p_value, _, _ = chi2_contingency([[n11, n10], [n01, n00]])
    return float(p_value)

if not rules_B.empty:
    rules_B["p_value"] = rules_B.apply(p_value_for_rule, axis=1)
else:
    rules_B["p_value"] = pd.Series(dtype=float)

rules_C = rules_B[rules_B["p_value"] < 0.05].copy()
removed_B = rules_A[rules_A["lift"] <= 1].copy()
removed_C = rules_B[rules_B["p_value"] >= 0.05].copy()

print(f"Chosen min_support={min_sup}, min_confidence={min_conf}")
print("All rules:", len(rules))
print("(A) Basic rules:", len(rules_A))
print("(B) Strong rules:", len(rules_B))
print("(C) Significant rules:", len(rules_C))

print("\nTop significant rules:")
if not rules_C.empty:
    cols = ["antecedents", "consequents", "support", "confidence", "lift", "p_value"]
    print(rules_C.sort_values("lift", ascending=False).head(20)[cols].to_string(index=False))
else:
    print("No significant rules.")

print("\nRules removed in B (lift <= 1):", len(removed_B))
if not removed_B.empty:
    print(removed_B[["antecedents", "consequents", "support", "confidence", "lift"]].head(10).to_string(index=False))

print("\nRules removed in C (p-value >= 0.05):", len(removed_C))
if not removed_C.empty:
    print(removed_C[["antecedents", "consequents", "support", "confidence", "lift", "p_value"]].head(10).to_string(index=False))

Chosen min_support=0.15, min_confidence=0.4


All rules: 40
(A) Basic rules: 40
(B) Strong rules: 40
(C) Significant rules: 36

Top significant rules:
                       antecedents                        consequents  support  confidence     lift      p_value
               frozenset({Eminem})                 frozenset({JAY Z}) 0.152174    0.777778 2.201709 9.066783e-09
                frozenset({JAY Z})                frozenset({Eminem}) 0.152174    0.430769 2.201709 9.066783e-09
                frozenset({JAY Z})            frozenset({Kanye West}) 0.228261    0.646154 2.085830 1.033685e-12
           frozenset({Kanye West})                 frozenset({JAY Z}) 0.228261    0.736842 2.085830 1.033685e-12
         frozenset({Lana Del Rey})            frozenset({Kanye West}) 0.173913    0.603774 1.949024 1.098866e-07
           frozenset({Kanye West})          frozenset({Lana Del Rey}) 0.173913    0.561404 1.949024 1.098866e-07
       frozenset({Arctic Monkeys})           frozenset({The Killers}) 0.168478    0.492063 1.886243 6.4